# Tyre RUL Modeling Notebook (Tree + Deep Models)

This notebook trains and compares:

1. XGBoost
2. LightGBM
3. CatBoost
4. MLP (Category Embedding)
5. TabTransformer
6. FT-Transformer

Target: `remaining_useful_life(km)`


## Environment Notes for Deep Models

Deep models in this notebook use **PyTorch Tabular**, which requires `torch` and `pytorch_tabular` to be installed in your `team_tyre` environment.

If deep imports fail, run the install commands provided in the final assistant response and re-run this notebook from the top.


In [3]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

SEED = 42
TARGET_COL = "remaining_useful_life(km)"
ARTIFACT_DIR = Path("artifacts")
PARQUET_PATH = ARTIFACT_DIR / "tyre_rul_cleaned.parquet"
CSV_PATH = ARTIFACT_DIR / "tyre_rul_cleaned.csv"

# Fast-iteration option for large-data experiments
USE_SAMPLE_FOR_FAST_EXPERIMENTS = False
SAMPLE_SIZE = 300_000

# Deep model training settings (safer defaults for 4GB VRAM laptops)
DEEP_MAX_EPOCHS = 25
DEEP_BATCH_SIZE = 512
DEEP_MAX_TRAIN_ROWS = 350_000

print("Core imports complete.")


Core imports complete.


In [4]:
HAS_DEEP_STACK = True
DEEP_IMPORT_ERROR = None

try:
    import torch
    from pytorch_tabular import TabularModel
    from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig
    from pytorch_tabular.models import (
        CategoryEmbeddingModelConfig,
        TabTransformerConfig,
        FTTransformerConfig,
    )
except Exception as ex:
    HAS_DEEP_STACK = False
    DEEP_IMPORT_ERROR = ex

if HAS_DEEP_STACK:
    print(f"Torch version: {torch.__version__}")
    print(f"CUDA available to torch: {torch.cuda.is_available()}")
else:
    print("Deep-learning stack is not available yet in this environment.")
    print(f"Import error: {DEEP_IMPORT_ERROR}")


W0505 21:19:57.737000 23140 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Torch version: 2.11.0+cu128
CUDA available to torch: True


In [5]:
if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    loaded_from = PARQUET_PATH
elif CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    loaded_from = CSV_PATH
else:
    raise FileNotFoundError("Could not find cleaned dataset in artifacts/.")

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column not found: {TARGET_COL}")

print(f"Loaded from: {loaded_from}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())


Loaded from: artifacts\tyre_rul_cleaned.parquet
Shape: 968,143 rows x 33 columns


,current_tread_depth(mm),Standard_tread_depth(mm),kilometers_driven(km),tread_wear_rating (UTQG),average_inflation_pressure(psi),recommended_inflation_pressure(psi),vehicle_sprung_mass(kg),vehicle_acceleration(0-100 km/h in seconds),maximum_power(hp),maximum_torque(N/m),maximum_speed (km/h),vehicle_mileage(mpg),average_tread_temperature(celsius),tyre_age(years),number_of_punctures,expected_tyre_life(km),retreaded,road_condition,weather_condition,axle_type(driven/dead),vehicle_model,fuel_type,transmission_type,tyre_brand,tyre_size,tread_material,country,remaining_useful_life(km),tread_depth_used_mm,tread_depth_used_pct,pressure_gap_psi,pressure_gap_pct,km_used_ratio_vs_expected
0,5.94,7.14,7500.0,300.0,22.10,35.0,1650.0,3.3,140.0,320.0,200.0,26.0,22.0,1.0,4.0,60000.0,0,smooth,tropical and humid,dead,mazda cx-5,petrol,manual,toyo a23,225/65r17,silica compound,brazil,32931.0,1.20,0.168067,-12.90,-0.368571,0.125000
1,1.56,6.35,57143.0,320.0,28.83,32.0,1575.0,5.0,190.0,400.0,250.0,30.0,15.0,5.0,11.0,80000.0,0,smooth,mild with distinct seasons,drive,bmw 3 series,petrol,automatic,bridgestone turanza t001,225/45r18,silica compound,japan,15819.0,4.79,0.754331,-3.17,-0.099063,0.714287
2,6.04,7.14,13571.0,500.0,29.72,35.0,1350.0,3.1,110.0,265.0,205.0,33.0,22.0,2.0,3.0,95000.0,0,smooth,tropical and humid,drive,hyundai elantra,diesel,manual,kumho solus ta31,205/55r16,silica and carbon black compound,brazil,66767.0,1.10,0.154062,-5.28,-0.150857,0.142853
3,3.64,7.14,21429.0,300.0,25.27,36.0,2241.0,11.6,500.0,854.0,322.0,396.0,-5.0,3.0,3.0,50000.0,0,offroad,cold,drive,tesla model s,electric,automatic,michelin pilot sport 4s,245/45r19,silica and carbon black compound,canada,19837.0,3.50,0.490196,-10.73,-0.298056,0.428580
4,1.73,7.94,43077.0,300.0,33.46,51.0,1600.0,3.7,194.0,376.0,210.0,26.0,10.0,7.0,7.0,80000.0,0,rough,mixed conditions,dead,subaru outback,petrol,manual,yokohama geolandar g91f,225/65r17,synthetic rubber,united kingdom,10198.0,6.21,0.782116,-17.54,-0.343922,0.538462


In [6]:
feature_cols = [c for c in df.columns if c != TARGET_COL]
categorical_cols = df[feature_cols].select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# Ensure categorical type is explicit for frameworks that can use native categorical handling
for col in categorical_cols:
    df[col] = df[col].astype("category")

X = df[feature_cols].copy()
y = df[TARGET_COL].astype("float32")

# 70/15/15 split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.1764705882, random_state=SEED
)

if USE_SAMPLE_FOR_FAST_EXPERIMENTS and len(X_train) > SAMPLE_SIZE:
    rng = np.random.default_rng(SEED)
    sampled_idx = rng.choice(X_train.index.to_numpy(), size=SAMPLE_SIZE, replace=False)
    X_train_fit = X_train.loc[sampled_idx].copy()
    y_train_fit = y_train.loc[sampled_idx].copy()
    print(f"Using sampled train set: {len(X_train_fit):,} rows")
else:
    X_train_fit = X_train.copy()
    y_train_fit = y_train.copy()
    print(f"Using full train set: {len(X_train_fit):,} rows")

print(f"Categorical features: {len(categorical_cols)}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Validation rows: {len(X_valid):,}")
print(f"Test rows: {len(X_test):,}")


Using full train set: 677,699 rows
Categorical features: 10
Numeric features: 22
Validation rows: 145,222
Test rows: 145,222


In [7]:
def regression_metrics(y_true, y_pred):
    return {
        "mae_km": float(mean_absolute_error(y_true, y_pred)),
        "rmse_km": float(root_mean_squared_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def evaluate_model(name, model, X_part, y_part, split_name):
    preds = model.predict(X_part)
    metrics = regression_metrics(y_part, preds)
    metrics.update({"model": name, "split": split_name})
    return metrics


def fit_with_gpu_fallback(build_model_fn, fit_fn, model_name):
    errors = []
    for mode in ("gpu", "cpu"):
        try:
            model = build_model_fn(mode)
            fit_fn(model)
            print(f"{model_name}: trained with {mode.upper()} mode")
            return model, mode
        except Exception as ex:
            errors.append((mode, str(ex)))
            print(f"{model_name}: {mode.upper()} mode failed -> {ex}")

    details = "\n".join([f"- {m}: {e}" for m, e in errors])
    raise RuntimeError(f"{model_name} failed in both GPU and CPU modes:\n{details}")


## Tree-Based Baselines


In [8]:
def build_xgb(mode):
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=1600,
        learning_rate=0.03,
        max_depth=10,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="hist",
        enable_categorical=True,
        device="cuda" if mode == "gpu" else "cpu",
        early_stopping_rounds=80,
        eval_metric="rmse",
        random_state=SEED,
        n_jobs=-1,
    )


def fit_xgb(model):
    model.fit(
        X_train_fit,
        y_train_fit,
        eval_set=[(X_valid, y_valid)],
        verbose=False,
    )


def build_lgb(mode):
    return lgb.LGBMRegressor(
        objective="regression",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=255,
        min_child_samples=50,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=0.5,
        random_state=SEED,
        n_jobs=-1,
        device_type="gpu" if mode == "gpu" else "cpu",
    )


def fit_lgb(model):
    model.fit(
        X_train_fit,
        y_train_fit,
        eval_set=[(X_valid, y_valid)],
        eval_metric="rmse",
        categorical_feature=categorical_cols,
        callbacks=[
            lgb.early_stopping(stopping_rounds=80),
            lgb.log_evaluation(period=250),
        ],
    )


def build_catboost(mode):
    params = dict(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=2500,
        learning_rate=0.03,
        depth=10,
        l2_leaf_reg=3.0,
        random_seed=SEED,
        od_type="Iter",
        od_wait=100,
        verbose=250,
        task_type="GPU" if mode == "gpu" else "CPU",
    )
    if mode == "gpu":
        params["devices"] = "0"
    return CatBoostRegressor(**params)


def fit_catboost(model):
    model.fit(
        X_train_fit,
        y_train_fit,
        eval_set=(X_valid, y_valid),
        cat_features=categorical_cols,
        use_best_model=True,
    )


xgb_model, xgb_mode = fit_with_gpu_fallback(build_xgb, fit_xgb, "XGBoost")
lgb_model, lgb_mode = fit_with_gpu_fallback(build_lgb, fit_lgb, "LightGBM")
cat_model, cat_mode = fit_with_gpu_fallback(build_catboost, fit_catboost, "CatBoost")


XGBoost: trained with GPU mode
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2287
[LightGBM] [Info] Number of data points in the train set: 677699, number of used features: 32
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 30 dense feature groups (20.68 MB) transferred to GPU in 0.022042 secs. 1 sparse feature groups
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 37807.930214
Training until validation scores don't improve for 80 rounds
[250]	valid_0's rmse: 279.183	valid_0's l2: 77943.1
[500]	valid_0's rmse: 196.651	valid_0's l2: 38671.8
[750]	valid_0's rmse: 170.045	valid_0's l2: 

## Deep Models (MLP, TabTransformer, FT-Transformer)


In [9]:
if not HAS_DEEP_STACK:
    print("Skipping deep models because torch/pytorch_tabular is not installed yet.")
    print("Install dependencies first, then rerun notebook from the top.")

deep_results_rows = []
deep_models = {}
deep_model_modes = {}

if HAS_DEEP_STACK:
    train_deep_df = X_train_fit.copy()
    train_deep_df[TARGET_COL] = y_train_fit.values

    valid_deep_df = X_valid.copy()
    valid_deep_df[TARGET_COL] = y_valid.values

    test_deep_df = X_test.copy()
    test_deep_df[TARGET_COL] = y_test.values

    if len(train_deep_df) > DEEP_MAX_TRAIN_ROWS:
        train_deep_df = train_deep_df.sample(DEEP_MAX_TRAIN_ROWS, random_state=SEED).copy()
        print(f"Deep training frame downsampled to: {len(train_deep_df):,} rows")
    else:
        print(f"Deep training frame size: {len(train_deep_df):,} rows")

    data_config = DataConfig(
        target=[TARGET_COL],
        continuous_cols=numeric_cols,
        categorical_cols=categorical_cols,
        normalize_continuous_features=True,
    )

    optimizer_config = OptimizerConfig()

    def get_prediction_column(pred_df):
        expected_col = f"{TARGET_COL}_prediction"
        if expected_col in pred_df.columns:
            return expected_col

        pred_cols = [c for c in pred_df.columns if c.endswith("_prediction")]
        if pred_cols:
            return pred_cols[0]

        raise ValueError("Prediction column not found in PyTorch Tabular output.")

    def train_deep_model(model_name, model_config):
        errors = []
        for mode in ("gpu", "cpu"):
            try:
                trainer_config = TrainerConfig(
                    auto_lr_find=False,
                    batch_size=DEEP_BATCH_SIZE,
                    max_epochs=DEEP_MAX_EPOCHS,
                    early_stopping="valid_loss",
                    early_stopping_patience=4,
                    accelerator="gpu" if mode == "gpu" else "cpu",
                    devices=1,
                )

                tab_model = TabularModel(
                    data_config=data_config,
                    model_config=model_config,
                    optimizer_config=optimizer_config,
                    trainer_config=trainer_config,
                )

                tab_model.fit(train=train_deep_df, validation=valid_deep_df)

                pred_valid = tab_model.predict(valid_deep_df)
                pred_test = tab_model.predict(test_deep_df)

                pred_col_valid = get_prediction_column(pred_valid)
                pred_col_test = get_prediction_column(pred_test)

                val_metrics = regression_metrics(valid_deep_df[TARGET_COL], pred_valid[pred_col_valid])
                test_metrics = regression_metrics(test_deep_df[TARGET_COL], pred_test[pred_col_test])

                deep_results_rows.append({
                    "model": model_name,
                    "trained_mode": mode,
                    "split": "validation",
                    **val_metrics,
                })
                deep_results_rows.append({
                    "model": model_name,
                    "trained_mode": mode,
                    "split": "test",
                    **test_metrics,
                })

                deep_models[model_name] = tab_model
                deep_model_modes[model_name] = mode
                print(f"{model_name}: trained with {mode.upper()} mode")
                return
            except Exception as ex:
                errors.append((mode, str(ex)))
                print(f"{model_name}: {mode.upper()} mode failed -> {ex}")

        details = "\n".join([f"- {m}: {e}" for m, e in errors])
        raise RuntimeError(f"{model_name} failed in both GPU and CPU modes:\n{details}")

    mlp_config = CategoryEmbeddingModelConfig(
        task="regression",
        layers="1024-512-256",
        activation="ReLU",
        learning_rate=1e-3,
    )

    tab_transformer_config = TabTransformerConfig(
        task="regression",
        learning_rate=7e-4,
    )

    ft_transformer_config = FTTransformerConfig(
        task="regression",
        learning_rate=7e-4,
    )

    train_deep_model("MLP_CategoryEmbedding", mlp_config)
    train_deep_model("TabTransformer", tab_transformer_config)
    train_deep_model("FTTransformer", ft_transformer_config)


2026-05-05 21:38:25,033 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-05-05 21:38:25,135 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders


Deep training frame downsampled to: 350,000 rows


2026-05-05 21:38:26,733 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-05-05 21:38:31,621 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-05-05 21:38:32,171 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-05-05 21:38:32,351 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Ti Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pyt

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  736 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │    964 │ train │     0 │
│ 2 │ head             │ LinearHead                │    257 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 737 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 737 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 26                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ FTTransformerBackbone │  271 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding2dLayer      │  5.2 K │ train │     0 │
│ 2 │ _head            │ LinearHead            │     33 │ train │     0 │
│ 3 │ loss             │ MSELoss               │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 276 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 276 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 142                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=25` reached.


2026-05-05 22:45:02,256 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-05-05 22:45:02,256 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


FTTransformer: trained with GPU mode


## Compare All Available Models


In [10]:
tree_models = {
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
    "CatBoost": cat_model,
}

tree_modes = {
    "XGBoost": xgb_mode,
    "LightGBM": lgb_mode,
    "CatBoost": cat_mode,
}

rows = []
for model_name, model in tree_models.items():
    val_scores = evaluate_model(model_name, model, X_valid, y_valid, "validation")
    test_scores = evaluate_model(model_name, model, X_test, y_test, "test")
    val_scores["trained_mode"] = tree_modes[model_name]
    test_scores["trained_mode"] = tree_modes[model_name]
    rows.extend([val_scores, test_scores])

rows.extend(deep_results_rows)

results_df = pd.DataFrame(rows)
results_df = results_df[["model", "trained_mode", "split", "mae_km", "rmse_km", "r2"]]

display(results_df.sort_values(["split", "rmse_km"]))

test_ranking = results_df[results_df["split"] == "test"].sort_values("rmse_km").reset_index(drop=True)
best_model_name = test_ranking.loc[0, "model"]

print("\nTest ranking (lower RMSE is better):")
display(test_ranking)
print(f"Best model on test split: {best_model_name}")


,model,trained_mode,split,mae_km,rmse_km,r2
3,LightGBM,gpu,test,97.856980,127.260231,0.999949
1,XGBoost,gpu,test,103.817673,136.209946,0.999941
5,CatBoost,gpu,test,109.508559,140.445011,0.999938
7,MLP_CategoryEmbedding,gpu,test,319.803619,436.961212,0.999396
9,TabTransformer,gpu,test,12788.300781,15757.830078,0.214091
11,FTTransformer,gpu,test,26220.464844,31025.140625,-2.046540
2,LightGBM,gpu,validation,97.527886,126.660350,0.999949
0,XGBoost,gpu,validation,103.934570,136.379059,0.999941
4,CatBoost,gpu,validation,109.456919,139.967864,0.999938
6,MLP_CategoryEmbedding,gpu,validation,319.583099,437.869019,0.999395



Test ranking (lower RMSE is better):


,model,trained_mode,split,mae_km,rmse_km,r2
0,LightGBM,gpu,test,97.856980,127.260231,0.999949
1,XGBoost,gpu,test,103.817673,136.209946,0.999941
2,CatBoost,gpu,test,109.508559,140.445011,0.999938
3,MLP_CategoryEmbedding,gpu,test,319.803619,436.961212,0.999396
4,TabTransformer,gpu,test,12788.300781,15757.830078,0.214091
5,FTTransformer,gpu,test,26220.464844,31025.140625,-2.046540


Best model on test split: LightGBM


In [11]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
results_path = ARTIFACT_DIR / "model_comparison_results_all.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved comparison results to: {results_path.resolve()}")


Saved comparison results to: C:\GitHub_Repos\team_Tyre\artifacts\model_comparison_results_all.csv


## Optional: Save Trained Models

Uncomment what you want to persist in `artifacts/trained_models/`.


In [12]:
model_dir = ARTIFACT_DIR / "trained_models"
model_dir.mkdir(parents=True, exist_ok=True)

# Tree models
xgb_model.save_model(model_dir / "xgboost_rul.json")
lgb_model.booster_.save_model(str(model_dir / "lightgbm_rul.txt"))
cat_model.save_model(str(model_dir / "catboost_rul.cbm"))

# Deep models (if trained)
if "MLP_CategoryEmbedding" in deep_models:
    deep_models["MLP_CategoryEmbedding"].save_model(str(model_dir / "mlp_category_embedding"))
if "TabTransformer" in deep_models:
    deep_models["TabTransformer"].save_model(str(model_dir / "tab_transformer"))
if "FTTransformer" in deep_models:
    deep_models["FTTransformer"].save_model(str(model_dir / "ft_transformer"))

print(f"Saved models to: {model_dir.resolve()}")


`weights_only` was not set, defaulting to `False`.
`weights_only` was not set, defaulting to `False`.
`weights_only` was not set, defaulting to `False`.


Saved models to: C:\GitHub_Repos\team_Tyre\artifacts\trained_models
